In [8]:
import pandas as pd

train_df = pd.read_parquet("chebi_dataset_train.parquet")
train_df

,mol_id,SMILES,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,...,class_490,class_491,class_492,class_493,class_494,class_495,class_496,class_497,class_498,class_499
0,mol_12500,CCCCC/C=C\CCCCCCCC(=O)O,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,mol_15962,Cc1cc2cc(O)cc(O)c2c(C)n1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,mol_42147,C[C@H](CCC[C@@H](C)C=O)[C@H]1CC[C@H]2[C@@H]3CC...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,mol_43459,CCN=C1C=CC(=C(c2ccc(NCC)cc2)c2ccc(NCC)c(C)c2)C=C1,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,mol_12734,C[C@H](CCC(O)=N[C@@H](Cc1c[nH]c2ccccc12)C(=O)O...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33663,mol_26342,C[C@H](OP(=O)(O)OC[C@@H](O)[C@@H](O)[C@@H](O)C...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
33664,mol_5233,O=C(Nc1ccc(Cl)cc1)Nc1ccc(Cl)cc1,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
33665,mol_36641,Cc1cn([C@H]2C[C@H](OP(=O)(O)OC[C@H]3O[C@@H](n4...,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
33666,mol_9633,O=C([O-])C(O)Cc1c[nH]c2ccccc12,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [9]:
test_df = pd.read_parquet("chebi_dataset_test_empty.parquet")
test_df

,mol_id,SMILES
0,mol_6861,OC[C@H]1O[C@H](O[C@H]2C(O)O[C@H](CO)[C@@H](O)[...
1,mol_29793,O.O=C([O-])CC(O)(CC(=O)[O-])C(=O)[O-].[K+].[K+...
2,mol_26953,CC(C)=CCC/C(C)=C/CC/C(C)=C/CC/C(C)=C\CC/C(C)=C...
3,mol_26053,C=C1CCC(C(C)C)CC1
4,mol_18653,O=C(O)C1CCCCC1=O
...,...,...
11218,mol_13480,Oc1cc2cc[nH]c2cc1O
11219,mol_42067,CNCC(=O)OCc1cccnc1N(C)C(=O)OC(C)[n+]1cnn(C[C@]...
11220,mol_43978,N[C@@H](CCC(=O)[O-])C(O)=Nc1ccc2ccccc2c1
11221,mol_16268,COc1cc(-c2[o+]c3cc(O)cc(O)c3cc2O[C@@H]2O[C@H](...


# Setup

In [10]:
%pip install scikit-fingerprints scikit-learn lightgbm -q


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Feature Engineering — SMILES → Fingerprint Vectors

We concatenate three complementary fingerprints:
- **ECFPFingerprint** (Morgan, radius=2, 2048 bits) — circular substructure features
- **MACCSFingerprint** (166 bits) — expert-curated substructure keys
- **RDKitFingerprint** (2048 bits) — path-based hashed features

In [11]:
import numpy as np
from skfp.fingerprints import ECFPFingerprint, MACCSFingerprint, RDKitFingerprint

def smiles_to_features(smiles_list: list[str], n_jobs: int = -1) -> np.ndarray:
    """Convert a list of SMILES strings to a concatenated fingerprint matrix."""
    ecfp   = ECFPFingerprint(fp_size=2048, radius=2, n_jobs=n_jobs)
    maccs  = MACCSFingerprint(n_jobs=n_jobs)
    rdkit  = RDKitFingerprint(fp_size=2048, n_jobs=n_jobs)

    X_ecfp  = ecfp.transform(smiles_list)
    X_maccs = maccs.transform(smiles_list)
    X_rdkit = rdkit.transform(smiles_list)

    return np.hstack([X_ecfp, X_maccs, X_rdkit])

LABEL_COLS = [c for c in train_df.columns if c.startswith("class_")]

print("Computing training fingerprints …")
X_train_full = smiles_to_features(train_df["SMILES"].tolist())
y_train_full = train_df[LABEL_COLS].values

print("Computing test fingerprints …")
X_test = smiles_to_features(test_df["SMILES"].tolist())

print(f"X_train shape: {X_train_full.shape}")
print(f"X_test  shape: {X_test.shape}")
print(f"Labels  shape: {y_train_full.shape}")

Computing training fingerprints …


[17:24:12] WARNING: not removing hydrogen atom without neighbors
[17:24:12] WARNING: not removing hydrogen atom without neighbors
[17:24:12] WARNING: not removing hydrogen atom without neighbors
[17:24:12] WARNING: not removing hydrogen atom without neighbors
[17:24:12] WARNING: not removing hydrogen atom without neighbors
[17:24:12] WARNING: not removing hydrogen atom without neighbors
[17:24:12] WARNING: not removing hydrogen atom without neighbors
[17:24:12] WARNING: not removing hydrogen atom without neighbors
[17:24:12] WARNING: not removing hydrogen atom without neighbors
[17:24:12] WARNING: not removing hydrogen atom without neighbors
[17:24:12] WARNING: not removing hydrogen atom without neighbors
[17:24:12] WARNING: not removing hydrogen atom without neighbors
[17:24:12] WARNING: not removing hydrogen atom without neighbors
[17:24:12] WARNING: not removing hydrogen atom without neighbors
[17:24:12] WARNING: not removing hydrogen atom without neighbors
[17:24:12] WARNING: not r

Computing test fingerprints …


[17:24:24] WARNING: not removing hydrogen atom without neighbors
[17:24:24] WARNING: not removing hydrogen atom without neighbors
[17:24:24] WARNING: not removing hydrogen atom without neighbors
[17:24:24] WARNING: not removing hydrogen atom without neighbors
[17:24:24] Unusual charge on atom 6 number of radical electrons set to zero
[17:24:24] WARNING: not removing hydrogen atom without neighbors
[17:24:24] WARNING: not removing hydrogen atom without neighbors
[17:24:24] WARNING: not removing hydrogen atom without neighbors
[17:24:24] WARNING: not removing hydrogen atom without neighbors
[17:24:24] WARNING: not removing hydrogen atom without neighbors
[17:24:24] WARNING: not removing hydrogen atom without neighbors
[17:24:24] WARNING: not removing hydrogen atom without neighbors
[17:24:24] WARNING: not removing hydrogen atom without neighbors
[17:24:24] WARNING: not removing hydrogen atom without neighbors
[17:24:24] WARNING: not removing hydrogen atom without neighbors
[17:24:24] WAR

X_train shape: (33668, 4262)
X_test  shape: (11223, 4262)
Labels  shape: (33668, 500)


# Model Training & Evaluation

Train/validation split (80/20), then fit a `MultiOutputClassifier(LGBMClassifier)`.
Metrics: micro-F1, macro-F1, and per-class F1 summary.

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    n_jobs=-1,
)

model = MultiOutputClassifier(lgbm, n_jobs=-1)
print("Fitting model …")
model.fit(X_train_full, y_train_full)
print("Done.")

Fitting model …
[LightGBM] [Info] Number of positive: 31057, number of negative: 2611
[LightGBM] [Info] Number of positive: 23406, number of negative: 10262
[LightGBM] [Info] Number of positive: 24252, number of negative: 9416
[LightGBM] [Info] Number of positive: 32795, number of negative: 873
[LightGBM] [Info] Number of positive: 30993, number of negative: 2675
[LightGBM] [Warning] Contains only one class
[LightGBM] [Info] Number of positive: 0, number of negative: 33668
[LightGBM] [Info] Number of positive: 33455, number of negative: 213
[LightGBM] [Info] Number of positive: 25025, number of negative: 8643
[LightGBM] [Info] Number of positive: 32401, number of negative: 1267
[LightGBM] [Info] Number of positive: 27709, number of negative: 5959
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.231211 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info]

# Test Set Predictions & Submission File

In [13]:
print("Predicting on test set …")
y_pred_test = model.predict(X_test)

submission_df = test_df[["mol_id", "SMILES"]].copy()
pred_df = pd.DataFrame(y_pred_test, columns=LABEL_COLS, dtype="int64")
submission_df = pd.concat([submission_df.reset_index(drop=True), pred_df], axis=1)

SUBMISSION_FILE = "submission3.parquet"
submission_df.to_parquet(SUBMISSION_FILE, index=False)
print(f"Saved {SUBMISSION_FILE}  shape={submission_df.shape}")
submission_df.head()

Predicting on test set …


/Users/blazejnowicki/miniconda3/envs/task1/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/blazejnowicki/miniconda3/envs/task1/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/blazejnowicki/miniconda3/envs/task1/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/blazejnowicki/miniconda3/envs/task1/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/blazejnowicki/miniconda3/envs/task1/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWa

Saved submission3.parquet  shape=(11223, 502)


,mol_id,SMILES,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,...,class_490,class_491,class_492,class_493,class_494,class_495,class_496,class_497,class_498,class_499
0,mol_6861,OC[C@H]1O[C@H](O[C@H]2C(O)O[C@H](CO)[C@@H](O)[...,1,1,1,1,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0
1,mol_29793,O.O=C([O-])CC(O)(CC(=O)[O-])C(=O)[O-].[K+].[K+...,1,1,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,mol_26953,CC(C)=CCC/C(C)=C/CC/C(C)=C/CC/C(C)=C\CC/C(C)=C...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,mol_26053,C=C1CCC(C(C)C)CC1,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,mol_18653,O=C(O)C1CCCCC1=O,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [14]:
import io
import os
import requests
from dotenv import load_dotenv

load_dotenv()

API_TOKEN = os.getenv("TEAM_TOKEN")
SERVER_URL = os.getenv("SERVER_URL")

buffer = io.BytesIO()
submission_df.to_parquet(buffer, index=False)
buffer.seek(0)

response = requests.post(
    f"{SERVER_URL}/task1",
    files={"parquet_file": buffer},
    headers={"X-API-Token": API_TOKEN},
)

try:
    data = response.json()
except Exception:
    data = response.text

print("Status:", response.status_code)
print("Response:", data)

Status: 200
Response: {'request_id': 'Tb8jVIisp7'}
